# 08 — Human Baseline: Episode Log & Itinerary Builder

**Purpose:** Plan a trip as a human expert using the travel world tools.
Every search and booking is automatically logged as a `ReActStep`, producing
an `EpisodeLog` in the same format as agent-generated episodes.
The resulting log is evaluated with the same metrics used to score the agent.

**Workflow:**
1. Run the setup cells (imports, config, session init).
2. Discover the world with `get_available_routes()` — note city IDs.
3. Search and book flights, hotels, attractions, and events.
4. Add each booking to the itinerary with `add_*_to_itinerary()`.
5. Check progress with `show_itinerary()` / `show_budget()` at any time.
6. Call `session.finalize()` and `session.save()` when done.
7. Run evaluation to see your scores.

> **Tip:** Each tool cell is independent — re-run any cell to retry with different arguments.
> Failed tool calls print an error and return `None`; the session continues normally.

---
## 1. Setup

In [1]:
import sys
from pathlib import Path

# Add project src to path (works whether you launch from repo root or notebooks/)
repo_root = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import json
import pandas as pd

from optimized_llm_planning_memory.core.models import UserRequest
from optimized_llm_planning_memory.evaluation.human_baseline import (
    HumanPlanningSession,
    display_routes,
    display_flights,
    display_hotels,
    display_attractions,
    display_restaurants,
    display_events,
    display_scores,
)

print('Imports OK')

Imports OK


### 1.1 Load a user request

Edit the path below to point to whichever request JSON you want to plan against.
The template request is a good starting point.

In [2]:
# ── CONFIGURE: change this path to the request you want to plan ──────────────
REQUEST_FILE = repo_root / 'data' / 'user_requests' / 'templates' / 'request_template.json'

# Optional: override any field before creating the session
# WORLD_SEED = 42  # set to match the agent episodes you want to compare against
WORLD_SEED = 42
OUTPUT_DIR = repo_root / 'outputs' / 'human_episodes'
# ─────────────────────────────────────────────────────────────────────────────

raw = json.loads(Path(REQUEST_FILE).read_text())
user_request = UserRequest.model_validate(raw)

print(f'Loaded request: {user_request.request_id}')
print(f'  {user_request.raw_text}')
print(f'  Budget: ${user_request.budget_usd:,.2f}')
print(f'  Dates : {user_request.start_date} → {user_request.end_date}')
print(f'  Hard constraints: {len(user_request.hard_constraints)}')
print(f'  Soft constraints: {len(user_request.soft_constraints)}')

Loaded request: TEMPLATE
  Plan a 7-day trip from Aeloria to Brindor and Caelum for 1 adult with a budget of $2500. I prefer boutique hotels, love art museums, and need vegetarian dining options.
  Budget: $2,500.00
  Dates : 2026-06-01 → 2026-06-07
  Hard constraints: 5
  Soft constraints: 3


### 1.2 Print the constraints (what you must satisfy)

In [3]:
print('HARD CONSTRAINTS (must satisfy):')
for c in user_request.hard_constraints:
    print(f'  [{c.category.value.upper()}]  {c.description}')

print('\nSOFT CONSTRAINTS (prefer to satisfy):')
for c in user_request.soft_constraints:
    print(f'  [{c.category.value.upper()}]  {c.description}')

print('\nPREFERENCES:')
for p in user_request.preferences:
    print(f'  • {p}')

HARD CONSTRAINTS (must satisfy):
  [BUDGET]  Total trip cost must not exceed $2500 USD.
  [DATE]  Trip must start on 2026-06-01 and end on 2026-06-07
  [DURATION]  Trip must be exactly 7 days long.
  [GROUP]  Party of 1 (1 adult(s)); all pricing and reservations must accommodate the full group.
  [ACTIVITY]  Itinerary must include at least one hiking activity.

SOFT CONSTRAINTS (prefer to satisfy):
  [ACCOMMODATION]  Hostels or budget hotels (1-2 stars) are acceptable.
  [ACTIVITY]  Include at least one outdoor or adventure activity per day.
  [PREFERENCE]  Favour local community experiences over tourist-trap attractions.

PREFERENCES:
  • hiking
  • local cuisine
  • cultural sites
  • photography
  • street art


### 1.3 Create the planning session

In [4]:
session = HumanPlanningSession(
    user_request=user_request,
    seed=WORLD_SEED,
    output_dir=OUTPUT_DIR,
    world_config={"num_cities_per_region": 3, "num_regions": 1}, 
)

╔══════════════════════════════════════════════════════════╗
║  Human Planning Session                                  ║
╠══════════════════════════════════════════════════════════╣
║  episode_id  : d7c11926131d                              ║
║  request_id  : TEMPLATE                                  ║
║  world seed  : 42                                        ║
║  budget      : $2,500.00                                 ║
║  dates       : 2026-06-01 → 2026-06-07                  ║
╚══════════════════════════════════════════════════════════╝

Request: Plan a 7-day trip from Aeloria to Brindor and Caelum for 1 adult with a budget of $2500. I prefer boutique hotels, love art museums, and need vegetarian dining options.



---
## 2. Discover the World

Always run this first — it reveals the city IDs required by every other tool.

In [5]:
routes = session.get_available_routes(
    thought='First I need to discover available cities and their IDs.'
)
display_routes(routes)

,city_name,city_id,connects_to
1,Aeloria,city_world_42_20260503_065531_0000,
2,Brindor,city_world_42_20260503_065531_0001,
3,Caelum,city_world_42_20260503_065531_0002,


In [8]:
# ── NOTE DOWN CITY IDs from the table above ───────────────────────────────────
# Fill these in from the routes output — they are used in every search call.
# Example: ORIGIN_ID = 'city_world_42...''  (whatever city_id appeared in the table)

ORIGIN_ID = 'city_world_42_20260503_065531_0000'          # e.g. 'city_world_42...'
DEST_1_ID = 'city_world_42_20260503_065531_0001'          # e.g. 'city_world_42...''
DEST_2_ID = ''          # e.g. 'city_world_42...''   (leave empty '' if only one destination)

# Dates (from user request — already set, but handy to have as variables)
START_DATE = user_request.start_date   # '2026-06-01'
END_DATE   = user_request.end_date     # '2026-06-07'
PASSENGERS = user_request.traveler_profile.total_travelers

print(f'Origin: {ORIGIN_ID}  |  Dest 1: {DEST_1_ID}  |  Dest 2: {DEST_2_ID}')
print(f'Dates : {START_DATE} → {END_DATE}  |  Passengers: {PASSENGERS}')

Origin: city_world_42_20260503_065531_0000  |  Dest 1: city_world_42_20260503_065531_0001  |  Dest 2: 
Dates : 2026-06-01 → 2026-06-07  |  Passengers: 1


---
## 3. Flights

Search → inspect results → select the best option → add to itinerary.

### 3.1 Outbound flight (Origin → Destination 1)

In [9]:
flights_out = session.search_flights(
    thought=f'Searching for flights from {ORIGIN_ID} to {DEST_1_ID} on {START_DATE}.',
    origin_city_id=ORIGIN_ID,
    destination_city_id=DEST_1_ID,
    departure_date=START_DATE,
    passengers=PASSENGERS,
)
display_flights(flights_out)

,edge_id,origin_city_id,destination_city_id,departure_datetime,arrival_datetime,total_price
1,edge_world_42_20260503_065531_0066,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T15:00:00,2026-06-01T19:27:00,144.03
2,edge_world_42_20260503_065531_0011,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T15:00:00,2026-06-01T16:55:00,144.21
3,edge_world_42_20260503_065531_0008,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T09:00:00,2026-06-01T10:10:00,165.79
4,edge_world_42_20260503_065531_0007,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T07:00:00,2026-06-01T09:00:00,183.42
5,edge_world_42_20260503_065531_0014,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T21:00:00,2026-06-01T22:15:00,207.16
6,edge_world_42_20260503_065531_0012,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T17:00:00,2026-06-01T18:45:00,211.94
7,edge_world_42_20260503_065531_0068,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T11:00:00,2026-06-01T15:27:00,236.58
8,edge_world_42_20260503_065531_0013,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T19:00:00,2026-06-01T21:00:00,237.53
9,edge_world_42_20260503_065531_0010,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T13:00:00,2026-06-01T14:55:00,239.90
10,edge_world_42_20260503_065531_0009,city_world_42_20260503_065531_0000,city_world_42_20260503_065531_0001,2026-06-01T11:00:00,2026-06-01T12:10:00,240.13


In [10]:
# ── SELECT AND BOOK ───────────────────────────────────────────────────────────
# Copy edge_id from the table above (row [1] is cheapest).

FLIGHT_OUT_EDGE_ID = flights_out[0]['edge_id'] if flights_out else ''

booking_flight_out = session.select_flight(
    thought=f'Taking the cheapest outbound flight on {START_DATE}.',
    edge_id=FLIGHT_OUT_EDGE_ID,
)
print(booking_flight_out)

{'booking_id': 'FLT-CF206312', 'edge_id': 'edge_world_42_20260503_065531_0066', 'origin_city_name': 'city_world_42_20260503_065531_0000', 'destination_city_name': 'city_world_42_20260503_065531_0001', 'departure_datetime': '2026-06-01T15:00:00', 'arrival_datetime': '2026-06-01T19:27:00', 'total_price': 144.03, 'status': 'selected', 'confirmation_datetime': '2026-05-03T06:56:30.228997+00:00'}


In [11]:
# Add flight to the itinerary
if booking_flight_out:
    session.add_flight_to_itinerary(
        date_str=START_DATE,
        booking=booking_flight_out,
        thought=f'Outbound leg: {ORIGIN_ID} → {DEST_1_ID} on {START_DATE}.',
    )

  ✅ Outbound leg: city_world_42_20260503_065531_0000 → city_world_42_20260503_065531_0001 on 2026-06-01.


### 3.2 Inter-destination flight (Destination 1 → Destination 2)

*Skip this section if you only have one destination.*

In [ ]:
# ── ADJUST: set the date you want to fly to destination 2 ────────────────────
INTER_DATE = ''   # e.g. '2026-06-04'

if DEST_2_ID and INTER_DATE:
    flights_inter = session.search_flights(
        thought=f'Searching flights from {DEST_1_ID} to {DEST_2_ID} on {INTER_DATE}.',
        origin_city_id=DEST_1_ID,
        destination_city_id=DEST_2_ID,
        departure_date=INTER_DATE,
        passengers=PASSENGERS,
    )
    display_flights(flights_inter)
else:
    print('Skipped (set DEST_2_ID and INTER_DATE to enable this cell)')
    flights_inter = []

In [ ]:
if flights_inter:
    booking_flight_inter = session.select_flight(
        thought=f'Selecting inter-destination flight on {INTER_DATE}.',
        edge_id=flights_inter[0]['edge_id'],
    )
    if booking_flight_inter:
        session.add_flight_to_itinerary(
            date_str=INTER_DATE,
            booking=booking_flight_inter,
            thought=f'Inter-destination leg: {DEST_1_ID} → {DEST_2_ID} on {INTER_DATE}.',
        )

### 3.3 Return flight

In [12]:
# ── ADJUST: which city are you flying home from? ──────────────────────────────
RETURN_FROM_ID = DEST_2_ID if DEST_2_ID else DEST_1_ID

flights_return = session.search_flights(
    thought=f'Searching return flights from {RETURN_FROM_ID} to {ORIGIN_ID} on {END_DATE}.',
    origin_city_id=RETURN_FROM_ID,
    destination_city_id=ORIGIN_ID,
    departure_date=END_DATE,
    passengers=PASSENGERS,
)
display_flights(flights_return)

,edge_id,origin_city_id,destination_city_id,departure_datetime,arrival_datetime,total_price
1,edge_world_42_20260503_065531_0031,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T15:00:00,2026-06-07T16:30:00,120.49
2,edge_world_42_20260503_065531_0074,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T15:00:00,2026-06-07T19:27:00,161.94
3,edge_world_42_20260503_065531_0030,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T13:00:00,2026-06-07T14:40:00,169.24
4,edge_world_42_20260503_065531_0033,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T19:00:00,2026-06-07T20:10:00,197.56
5,edge_world_42_20260503_065531_0029,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T11:00:00,2026-06-07T12:50:00,198.10
6,edge_world_42_20260503_065531_0032,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T17:00:00,2026-06-07T18:25:00,204.21
7,edge_world_42_20260503_065531_0028,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T09:00:00,2026-06-07T10:25:00,213.17
8,edge_world_42_20260503_065531_0074,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T15:00:00,2026-06-07T19:27:00,215.67
9,edge_world_42_20260503_065531_0027,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T07:00:00,2026-06-07T08:25:00,216.19
10,edge_world_42_20260503_065531_0072,city_world_42_20260503_065531_0001,city_world_42_20260503_065531_0000,2026-06-07T19:00:00,2026-06-07T23:27:00,221.15


In [13]:
if flights_return:
    booking_flight_return = session.select_flight(
        thought=f'Taking the return flight on {END_DATE}.',
        edge_id=flights_return[0]['edge_id'],
    )
    if booking_flight_return:
        session.add_flight_to_itinerary(
            date_str=END_DATE,
            booking=booking_flight_return,
            thought=f'Return leg: {RETURN_FROM_ID} → {ORIGIN_ID} on {END_DATE}.',
        )

  ✅ Return leg: city_world_42_20260503_065531_0001 → city_world_42_20260503_065531_0000 on 2026-06-07.


---
## 4. Hotels

Book accommodation for each destination. Tip: set `max_price_per_night` to
`session.budget_remaining / remaining_nights` to filter out unaffordable options.

### 4.1 Hotel at Destination 1

In [ ]:
# ── ADJUST: check-in and check-out dates for this destination ─────────────────
HOTEL_1_CHECK_IN  = START_DATE          # first night at destination 1
HOTEL_1_CHECK_OUT = INTER_DATE if INTER_DATE else END_DATE

hotels_1 = session.search_hotels(
    thought=f'Searching hotels in {DEST_1_ID} for {HOTEL_1_CHECK_IN} → {HOTEL_1_CHECK_OUT}.',
    city_id=DEST_1_ID,
    check_in=HOTEL_1_CHECK_IN,
    check_out=HOTEL_1_CHECK_OUT,
    guests=PASSENGERS,
    max_price_per_night=session.budget_remaining / max(1, 3),  # rough estimate
)
display_hotels(hotels_1)

In [ ]:
if hotels_1:
    HOTEL_1_ID = hotels_1[0]['hotel_id']   # pick from the table above

    booking_hotel_1 = session.book_hotel(
        thought=f'Booking the top-rated affordable hotel in {DEST_1_ID}.',
        hotel_id=HOTEL_1_ID,
        check_in=HOTEL_1_CHECK_IN,
        check_out=HOTEL_1_CHECK_OUT,
    )
    print(booking_hotel_1)

In [ ]:
if hotels_1 and booking_hotel_1:
    session.add_hotel_to_itinerary(
        booking=booking_hotel_1,
        city=DEST_1_ID,
        # Provide these if the booking dict doesn't have them
        hotel_name=hotels_1[0].get('name', ''),
        star_rating=hotels_1[0].get('star_rating'),
        cost_per_night=hotels_1[0].get('price_per_night'),
        thought=f'Accommodation in {DEST_1_ID} for {HOTEL_1_CHECK_IN} → {HOTEL_1_CHECK_OUT}.',
    )

### 4.2 Hotel at Destination 2 (skip if only one destination)

In [ ]:
HOTEL_2_CHECK_IN  = INTER_DATE if INTER_DATE else ''
HOTEL_2_CHECK_OUT = END_DATE

if DEST_2_ID and HOTEL_2_CHECK_IN:
    hotels_2 = session.search_hotels(
        thought=f'Searching hotels in {DEST_2_ID} for {HOTEL_2_CHECK_IN} → {HOTEL_2_CHECK_OUT}.',
        city_id=DEST_2_ID,
        check_in=HOTEL_2_CHECK_IN,
        check_out=HOTEL_2_CHECK_OUT,
        guests=PASSENGERS,
        max_price_per_night=session.budget_remaining / max(1, 3),
    )
    display_hotels(hotels_2)
else:
    print('Skipped')
    hotels_2 = []

In [ ]:
if hotels_2:
    HOTEL_2_ID = hotels_2[0]['hotel_id']
    booking_hotel_2 = session.book_hotel(
        thought=f'Booking hotel in {DEST_2_ID}.',
        hotel_id=HOTEL_2_ID,
        check_in=HOTEL_2_CHECK_IN,
        check_out=HOTEL_2_CHECK_OUT,
    )
    if booking_hotel_2:
        session.add_hotel_to_itinerary(
            booking=booking_hotel_2,
            city=DEST_2_ID,
            hotel_name=hotels_2[0].get('name', ''),
            star_rating=hotels_2[0].get('star_rating'),
            cost_per_night=hotels_2[0].get('price_per_night'),
            thought=f'Accommodation in {DEST_2_ID} for {HOTEL_2_CHECK_IN} → {HOTEL_2_CHECK_OUT}.',
        )

---
## 5. Activities & Attractions

Search for things to do in each destination. Add the ones you want to each day.
Duplicate or modify these cells for each activity.

### 5.1 Attractions at Destination 1

In [ ]:
# Search all categories first to get an overview
attractions_1 = session.search_attractions(
    thought=f'Exploring what attractions are available in {DEST_1_ID}.',
    city_id=DEST_1_ID,
)
display_attractions(attractions_1)

In [ ]:
# Search specifically for hiking (required hard constraint)
hiking_1 = session.search_attractions(
    thought=f'Looking for hiking options in {DEST_1_ID} — required by hard constraint.',
    city_id=DEST_1_ID,
    category='hiking',
)
display_attractions(hiking_1)

In [ ]:
# ── ADD A HIKING ACTIVITY ─────────────────────────────────────────────────────
# Edit the values below from the hiking_1 table above.

if hiking_1:
    chosen_hike = hiking_1[0]  # pick from the table

    # Optionally get full details first
    # detail = session.get_attraction_detail(
    #     thought='Checking hike details before committing.',
    #     attraction_id=chosen_hike['attraction_id'],
    # )

    session.add_activity_to_itinerary(
        date_str=START_DATE,          # which day to schedule this
        city=DEST_1_ID,
        activity_id=chosen_hike.get('attraction_id', ''),
        activity_name=chosen_hike.get('name', 'Hiking Trail'),
        location=chosen_hike.get('location', DEST_1_ID),
        start_time='09:00',
        duration_hours=chosen_hike.get('duration_hours', 4.0),
        cost_usd=chosen_hike.get('admission_fee', 0.0) * PASSENGERS,
        category='hiking',
        thought='Adding required hiking activity — satisfies hard constraint.',
    )

In [ ]:
# ── ADD MORE ACTIVITIES ───────────────────────────────────────────────────────
# Copy this pattern for each additional activity you want to include.

# Example: add a museum visit on day 2
# from datetime import date, timedelta
# day2 = str(date.fromisoformat(START_DATE) + timedelta(days=1))
#
# museums = session.search_attractions(
#     thought='Looking for museums in destination 1.',
#     city_id=DEST_1_ID,
#     category='museum',
# )
# display_attractions(museums)
#
# if museums:
#     session.add_activity_to_itinerary(
#         date_str=day2,
#         city=DEST_1_ID,
#         activity_id=museums[0]['attraction_id'],
#         activity_name=museums[0]['name'],
#         start_time='14:00',
#         duration_hours=2.5,
#         cost_usd=museums[0].get('admission_fee', 0.0) * PASSENGERS,
#         category='museum',
#         thought='Afternoon museum visit on day 2.',
#     )

print('Add more activities by copying and uncommenting the block above.')

### 5.2 Attractions at Destination 2 (skip if only one destination)

In [ ]:
if DEST_2_ID:
    attractions_2 = session.search_attractions(
        thought=f'Exploring attractions in {DEST_2_ID}.',
        city_id=DEST_2_ID,
    )
    display_attractions(attractions_2)
else:
    print('Skipped')
    attractions_2 = []

---
## 6. Restaurants

Restaurants are informational — no booking required.
Add them as ActivityBookings with `category='restaurant'`.

In [ ]:
restaurants_1 = session.search_restaurants(
    thought=f'Looking for vegetarian-friendly restaurants in {DEST_1_ID}.',
    city_id=DEST_1_ID,
    # cuisine='vegetarian',          # uncomment to filter by cuisine
    # max_avg_spend=30.0,            # uncomment to cap per-person spend
)
display_restaurants(restaurants_1)

In [ ]:
# ── ADD A RESTAURANT VISIT ────────────────────────────────────────────────────
# Restaurants are not bookable — just record the visit as an activity.

if restaurants_1:
    chosen_resto = restaurants_1[0]
    avg_spend = chosen_resto.get('avg_spend_per_person', 25.0)

    from datetime import date as _date, timedelta
    day2 = str(_date.fromisoformat(START_DATE) + timedelta(days=1))

    session.add_activity_to_itinerary(
        date_str=day2,
        city=DEST_1_ID,
        activity_id=chosen_resto.get('restaurant_id', ''),
        activity_name=chosen_resto.get('name', 'Restaurant'),
        location=chosen_resto.get('location', DEST_1_ID),
        start_time='19:00',
        duration_hours=1.5,
        cost_usd=avg_spend * PASSENGERS,
        category='restaurant',
        thought='Dinner at a local restaurant — satisfies vegetarian preference.',
    )

---
## 7. Events (Optional)

Book ticketed events that fall within the trip dates.

In [ ]:
events_1 = session.search_events(
    thought=f'Checking for events in {DEST_1_ID} during the trip.',
    city_id=DEST_1_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    max_price=session.budget_remaining * 0.1,  # events shouldn't blow the budget
)
display_events(events_1)

In [ ]:
# ── BOOK AN EVENT ─────────────────────────────────────────────────────────────
# Uncomment and edit if you want to book an event.

# if events_1:
#     chosen_event = events_1[0]
#     booking_event = session.book_event(
#         thought=f'Booking event: {chosen_event["name"]}.',
#         event_id=chosen_event['event_id'],
#         quantity=PASSENGERS,
#     )
#     if booking_event:
#         session.add_event_to_itinerary(
#             booking=booking_event,
#             city=DEST_1_ID,
#             event_name=chosen_event.get('name', ''),
#             thought='Added event booking to itinerary.',
#         )

print('Uncomment the block above to book an event.')

---
## 8. Free-form Planning Cells

Use these template cells to add any additional steps — extra searches, route
planning, additional activities, etc. Duplicate as needed.

In [ ]:
# ── TEMPLATE: call any tool ───────────────────────────────────────────────────
# result = session.search_attractions(
#     thought='...',
#     city_id=DEST_1_ID,
#     category='...',
# )
# display_attractions(result)

# ── TEMPLATE: add activity ────────────────────────────────────────────────────
# session.add_activity_to_itinerary(
#     date_str='YYYY-MM-DD',
#     city=DEST_1_ID,
#     activity_id='...',
#     activity_name='...',
#     start_time='10:00',
#     duration_hours=2.0,
#     cost_usd=0.0,
#     category='...',
#     thought='Reason for choosing this activity.',
# )

print('Template cell — add your own calls here.')

In [ ]:
# ── TEMPLATE: route planning ──────────────────────────────────────────────────
# routes_within = session.plan_route(
#     thought='Checking how to get from hotel to the hiking trail.',
#     origin_location_id='...',
#     destination_location_id='...',
#     departure_datetime='2026-06-01T09:00:00',
#     modes=['taxi', 'walk'],
# )
# print(routes_within)

print('Template cell — add your own route planning here.')

---
## 9. Progress Check

Run these at any time to inspect the current state of your plan.

In [ ]:
session.show_itinerary()

In [ ]:
session.show_budget()

In [ ]:
session.show_trail()

---
## 10. Finalize & Save

When you're satisfied with the itinerary, finalize and save the episode log.
Use `'HUMAN_INCOMPLETE'` if you stopped early.

In [ ]:
episode_log = session.finalize(
    termination_reason='DONE_ITINERARY',   # or 'HUMAN_INCOMPLETE'
)

In [ ]:
saved_path = session.save()
print(f'Episode log saved to: {saved_path}')

---
## 11. Evaluation

Score the human plan using the same deterministic metrics applied to agent episodes.

In [ ]:
from optimized_llm_planning_memory.evaluation.deterministic import DeterministicEvaluator

evaluator = DeterministicEvaluator()
scores = evaluator.score(episode_log, user_request)

display_scores(scores)

In [ ]:
# Detailed constraint satisfaction breakdown
from optimized_llm_planning_memory.core.constraints import ConstraintSatisfactionEngine

engine = ConstraintSatisfactionEngine()
all_constraints = list(user_request.hard_constraints) + list(user_request.soft_constraints)
results = engine.evaluate(episode_log.final_itinerary, all_constraints)

print('CONSTRAINT SATISFACTION DETAIL')
print('─' * 70)
for r in results:
    c = next((c for c in all_constraints if c.constraint_id == r.constraint_id), None)
    c_type = c.constraint_type.value.upper() if c else '?'
    icon = '✓' if r.satisfied else '✗'
    print(f'  {icon} [{c_type}]  score={r.score:.2f}  {r.constraint_id}')
    print(f'       {r.explanation}')

---
## 12. Inspect the Episode Log

Verify the structure of the saved log matches the agent format.

In [ ]:
print(f'episode_id      : {episode_log.episode_id}')
print(f'agent_mode      : {episode_log.agent_mode}')
print(f'world_seed      : {episode_log.world_seed}')
print(f'total_steps     : {episode_log.total_steps}')
print(f'success         : {episode_log.success}')
print(f'termination     : {episode_log.termination_reason}')
print(f'days planned    : {len(episode_log.final_itinerary.days)}')
print(f'total_cost_usd  : ${episode_log.final_itinerary.total_cost_usd:.2f}')
print(f'tool_stats      : {len(episode_log.tool_stats)} tools used')
print()
print('Reward components:')
rc = episode_log.reward_components
print(f'  hard_constraint_score   : {rc.hard_constraint_score:.3f}')
print(f'  soft_constraint_score   : {rc.soft_constraint_score:.3f}')
print(f'  tool_efficiency_score   : {rc.tool_efficiency_score:.3f}')
print(f'  logical_consistency     : {rc.logical_consistency_score:.3f}')
print(f'  total_reward            : {rc.total_reward:.3f}')

In [ ]:
# Print the full trajectory as text (same format the agent and compressor see)
print(episode_log.trajectory.to_text(include_itinerary_snapshots=False))

In [ ]:
# Load the saved JSON and verify it round-trips correctly
from optimized_llm_planning_memory.utils.episode_io import load_episode

reloaded = load_episode(saved_path)
assert reloaded.episode_id == episode_log.episode_id
assert reloaded.agent_mode == 'human_baseline'
print(f'Episode round-trip OK — {saved_path.name}')